# 7. Train and compare pose classification models

直接使用已切分好的 train / validation / test，不再呼叫 `train_test_split`。以 validation macro-F1 選擇最佳模型，並保存所有模型、逐筆各類別預測分數、整體與各姿勢評估指標，以及 confusion matrix。

In [1]:
from pathlib import Path
import json
import re
import time
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.ensemble import RandomForestClassifier
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    precision_recall_fscore_support,
)
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

RANDOM_STATE = 42
TARGET_COLUMN = 'label'
SAMPLE_COLUMN = 'image_name'
SELECTION_METRIC = 'macro_f1'


def find_project_root(start: Path = Path.cwd()) -> Path:
    for directory in (start.resolve(), *start.resolve().parents):
        candidate = directory / 'csv_data' / 'prepared_to_train' / 'keypoints_train.csv'
        if candidate.is_file():
            return directory
    raise FileNotFoundError('Could not find csv_data/prepared_to_train/keypoints_train.csv')


PROJECT_ROOT = find_project_root()
INPUT_DIR = PROJECT_ROOT / 'csv_data' / 'prepared_to_train'
OUTPUT_DIR = PROJECT_ROOT / 'model' / 'code7_pose_classification'
MODEL_DIR = OUTPUT_DIR / 'models'
PREDICTION_DIR = OUTPUT_DIR / 'predictions'
CONFUSION_DIR = OUTPUT_DIR / 'confusion_matrices'
for directory in (OUTPUT_DIR, MODEL_DIR, PREDICTION_DIR, CONFUSION_DIR):
    directory.mkdir(parents=True, exist_ok=True)

INPUT_FILES = {
    'train': INPUT_DIR / 'keypoints_train.csv',
    'validation': INPUT_DIR / 'keypoints_validation.csv',
    'test': INPUT_DIR / 'keypoints_test_dbscan_subposes.csv',
}
PROJECT_ROOT


WindowsPath('C:/Users/vgohu/Desktop/yoga_project')

In [2]:
datasets = {split: pd.read_csv(path) for split, path in INPUT_FILES.items()}
FEATURE_COLUMNS = [
    column for column in datasets['train'].columns
    if column.startswith('kp_') or column.endswith('_angle_deg')
]
if not FEATURE_COLUMNS:
    raise ValueError('No keypoint or angle feature columns were found.')

train_columns = set(datasets['train'].columns)
for split, dataframe in datasets.items():
    missing = {SAMPLE_COLUMN, TARGET_COLUMN, *FEATURE_COLUMNS}.difference(dataframe.columns)
    if missing:
        raise ValueError(f'{split} is missing columns: {sorted(missing)}')
    values = dataframe[FEATURE_COLUMNS].to_numpy(dtype=float)
    if not np.isfinite(values).all():
        raise ValueError(f'{split} feature data contains NaN or infinite values.')
    if dataframe[SAMPLE_COLUMN].duplicated().any():
        raise ValueError(f'{split} contains duplicate sample paths.')

CLASSES = sorted(datasets['train'][TARGET_COLUMN].astype(str).unique())
for split in ('validation', 'test'):
    unknown = set(datasets[split][TARGET_COLUMN].astype(str)).difference(CLASSES)
    if unknown:
        raise ValueError(f'{split} contains labels absent from training: {sorted(unknown)}')

X = {split: df[FEATURE_COLUMNS] for split, df in datasets.items()}
y = {split: df[TARGET_COLUMN].astype(str) for split, df in datasets.items()}
pd.DataFrame({
    split: y[split].value_counts().reindex(CLASSES, fill_value=0)
    for split in datasets
}).rename_axis('pose')


,train,validation,test
pose,,,
downdog,159,40,94
goddess,130,32,80
plank,204,51,115
tree,120,30,69
warrior2,199,50,107


In [3]:
models = {
    'mlp': Pipeline([
        ('scaler', StandardScaler()),
        ('classifier', MLPClassifier(
            hidden_layer_sizes=(128, 64), activation='relu', solver='adam',
            alpha=1e-4, batch_size=32, learning_rate_init=1e-3,
            max_iter=1500, random_state=RANDOM_STATE, early_stopping=False,
        )),
    ]),
    'logistic_regression': Pipeline([
        ('scaler', StandardScaler()),
        ('classifier', LogisticRegression(
            C=1.0, max_iter=3000, class_weight='balanced',
            random_state=RANDOM_STATE,
        )),
    ]),
    'random_forest': RandomForestClassifier(
        n_estimators=500, max_features='sqrt', min_samples_leaf=1,
        class_weight='balanced', n_jobs=-1, random_state=RANDOM_STATE,
    ),
    'rbf_svm': Pipeline([
        ('scaler', StandardScaler()),
        ('classifier', SVC(
            C=3.0, kernel='rbf', gamma='scale', probability=True,
            class_weight='balanced', random_state=RANDOM_STATE,
        )),
    ]),
}
list(models)


['mlp', 'logistic_regression', 'random_forest', 'rbf_svm']

In [4]:
def safe_name(value: str) -> str:
    return re.sub(r'[^0-9A-Za-z_-]+', '_', str(value)).strip('_')


def aligned_probabilities(model, features: pd.DataFrame) -> np.ndarray:
    raw = model.predict_proba(features)
    model_classes = [str(value) for value in model.classes_]
    indices = [model_classes.index(pose) for pose in CLASSES]
    return raw[:, indices]


def save_confusion_figure(matrix: np.ndarray, title: str, path: Path) -> None:
    fig, ax = plt.subplots(figsize=(7, 6))
    image = ax.imshow(matrix, cmap='Blues')
    fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
    ax.set(xticks=range(len(CLASSES)), yticks=range(len(CLASSES)),
           xticklabels=CLASSES, yticklabels=CLASSES,
           xlabel='Predicted pose', ylabel='True pose', title=title)
    plt.setp(ax.get_xticklabels(), rotation=35, ha='right')
    threshold = matrix.max() / 2 if matrix.size else 0
    for row in range(matrix.shape[0]):
        for column in range(matrix.shape[1]):
            ax.text(column, row, int(matrix[row, column]), ha='center', va='center',
                    color='white' if matrix[row, column] > threshold else 'black')
    fig.tight_layout()
    fig.savefig(path, dpi=180, bbox_inches='tight')
    plt.close(fig)


overall_rows = []
pose_rows = []
confusion_rows = []
fit_rows = []

for model_name, model in models.items():
    print(f'Training {model_name} ...')
    started = time.perf_counter()
    with warnings.catch_warnings():
        warnings.filterwarnings('always', category=ConvergenceWarning)
        model.fit(X['train'], y['train'])
    fit_seconds = time.perf_counter() - started
    model_path = MODEL_DIR / f'{model_name}.joblib'
    joblib.dump(model, model_path)
    fit_rows.append({'model': model_name, 'fit_seconds': fit_seconds, 'model_file': str(model_path)})

    for split in ('validation', 'test'):
        predicted = model.predict(X[split]).astype(str)
        probabilities = aligned_probabilities(model, X[split])
        precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
            y[split], predicted, labels=CLASSES, average='macro', zero_division=0
        )
        precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(
            y[split], predicted, labels=CLASSES, average='weighted', zero_division=0
        )
        overall_rows.append({
            'model': model_name, 'split': split, 'samples': len(y[split]),
            'accuracy': accuracy_score(y[split], predicted),
            'macro_precision': precision_macro, 'macro_recall': recall_macro,
            'macro_f1': f1_macro, 'weighted_precision': precision_weighted,
            'weighted_recall': recall_weighted, 'weighted_f1': f1_weighted,
        })

        report = classification_report(
            y[split], predicted, labels=CLASSES, target_names=CLASSES,
            output_dict=True, zero_division=0,
        )
        for pose in CLASSES:
            pose_rows.append({
                'model': model_name, 'split': split, 'pose': pose,
                'precision': report[pose]['precision'], 'recall': report[pose]['recall'],
                'f1': report[pose]['f1-score'], 'support': int(report[pose]['support']),
            })

        matrix = confusion_matrix(y[split], predicted, labels=CLASSES)
        for true_index, true_pose in enumerate(CLASSES):
            for predicted_index, predicted_pose in enumerate(CLASSES):
                confusion_rows.append({
                    'model': model_name, 'split': split, 'true_pose': true_pose,
                    'predicted_pose': predicted_pose,
                    'count': int(matrix[true_index, predicted_index]),
                })
        save_confusion_figure(
            matrix, f'{model_name} - {split}',
            CONFUSION_DIR / f'{model_name}_{split}_confusion_matrix.png',
        )

        prediction_table = pd.DataFrame({
            'sample': datasets[split][SAMPLE_COLUMN].astype(str),
            'true_label': y[split], 'predicted_label': predicted,
            'is_correct': y[split].to_numpy() == predicted,
        })
        for class_index, pose in enumerate(CLASSES):
            prediction_table[f'score_{safe_name(pose)}'] = probabilities[:, class_index]
        prediction_table.to_csv(
            PREDICTION_DIR / f'{model_name}_{split}_predictions.csv', index=False
        )

overall_metrics = pd.DataFrame(overall_rows).sort_values(['split', 'macro_f1'], ascending=[True, False])
per_pose_metrics = pd.DataFrame(pose_rows).sort_values(['split', 'model', 'pose'])
confusion_long = pd.DataFrame(confusion_rows)
training_times = pd.DataFrame(fit_rows).sort_values('fit_seconds')

overall_metrics.to_csv(OUTPUT_DIR / 'overall_metrics.csv', index=False)
per_pose_metrics.to_csv(OUTPUT_DIR / 'classification_report_per_pose.csv', index=False)
confusion_long.to_csv(OUTPUT_DIR / 'confusion_matrices.csv', index=False)
training_times.to_csv(OUTPUT_DIR / 'training_times.csv', index=False)
display(overall_metrics)
display(per_pose_metrics)


Training mlp ...
Training logistic_regression ...
Training random_forest ...
Training rbf_svm ...


,model,split,samples,accuracy,macro_precision,macro_recall,macro_f1,weighted_precision,weighted_recall,weighted_f1
5,random_forest,test,465,0.984946,0.986604,0.982872,0.984359,0.985317,0.984946,0.984783
7,rbf_svm,test,465,0.978495,0.979436,0.978156,0.978690,0.978571,0.978495,0.978436
1,mlp,test,465,0.969892,0.970855,0.970438,0.970402,0.970138,0.969892,0.969785
3,logistic_regression,test,465,0.967742,0.969653,0.967800,0.968135,0.968661,0.967742,0.967617
4,random_forest,validation,203,0.970443,0.974594,0.966000,0.969201,0.971616,0.970443,0.970080
6,rbf_svm,validation,203,0.960591,0.967195,0.955828,0.959874,0.962413,0.960591,0.960102
0,mlp,validation,203,0.950739,0.960192,0.945657,0.950563,0.953616,0.950739,0.950156
2,logistic_regression,validation,203,0.945813,0.954143,0.939407,0.943518,0.949147,0.945813,0.944760


,model,split,pose,precision,recall,f1,support
15,logistic_regression,test,downdog,0.978947,0.989362,0.984127,94
16,logistic_regression,test,goddess,1.000000,0.925000,0.961039,80
17,logistic_regression,test,plank,0.972973,0.939130,0.955752,115
18,logistic_regression,test,tree,0.957746,0.985507,0.971429,69
19,logistic_regression,test,warrior2,0.938596,1.000000,0.968326,107
5,mlp,test,downdog,0.989362,0.989362,0.989362,94
6,mlp,test,goddess,0.973684,0.925000,0.948718,80
7,mlp,test,plank,0.973451,0.956522,0.964912,115
8,mlp,test,tree,0.971831,1.000000,0.985714,69
9,mlp,test,warrior2,0.945946,0.981308,0.963303,107


In [5]:
validation_ranking = (
    overall_metrics[overall_metrics['split'].eq('validation')]
    .sort_values([SELECTION_METRIC, 'accuracy'], ascending=False)
    .reset_index(drop=True)
)
best_model_name = validation_ranking.loc[0, 'model']
best_model = models[best_model_name]
best_model_path = OUTPUT_DIR / 'best_model.joblib'
joblib.dump(best_model, best_model_path)

manifest = {
    'best_model': best_model_name,
    'selection_split': 'validation',
    'selection_metric': SELECTION_METRIC,
    'selection_value': float(validation_ranking.loc[0, SELECTION_METRIC]),
    'target_column': TARGET_COLUMN,
    'sample_column': SAMPLE_COLUMN,
    'classes': CLASSES,
    'feature_columns': FEATURE_COLUMNS,
    'input_files': {key: str(value) for key, value in INPUT_FILES.items()},
    'best_model_file': str(best_model_path),
    'note': 'Model selection used validation macro-F1 only; test was not used to select a model.',
}
with (OUTPUT_DIR / 'model_manifest.json').open('w', encoding='utf-8') as file:
    json.dump(manifest, file, ensure_ascii=False, indent=2)
validation_ranking.to_csv(OUTPUT_DIR / 'validation_model_ranking.csv', index=False)

required_outputs = [
    best_model_path, OUTPUT_DIR / 'overall_metrics.csv',
    OUTPUT_DIR / 'classification_report_per_pose.csv',
    OUTPUT_DIR / 'confusion_matrices.csv', OUTPUT_DIR / 'model_manifest.json',
]
assert all(path.is_file() for path in required_outputs)
assert len(list(MODEL_DIR.glob('*.joblib'))) == len(models)
assert len(list(PREDICTION_DIR.glob('*_predictions.csv'))) == 2 * len(models)
assert len(list(CONFUSION_DIR.glob('*.png'))) == 2 * len(models)
assert np.allclose(
    pd.read_csv(PREDICTION_DIR / f'{best_model_name}_test_predictions.csv')
      [[f'score_{safe_name(pose)}' for pose in CLASSES]].sum(axis=1),
    1.0,
)

print(f'Best model: {best_model_name}')
print(f'Validation macro-F1: {validation_ranking.loc[0, SELECTION_METRIC]:.4f}')
print(f'All outputs saved to: {OUTPUT_DIR}')
display(validation_ranking)


Best model: random_forest
Validation macro-F1: 0.9692
All outputs saved to: C:\Users\vgohu\Desktop\yoga_project\model\code7_pose_classification


,model,split,samples,accuracy,macro_precision,macro_recall,macro_f1,weighted_precision,weighted_recall,weighted_f1
0,random_forest,validation,203,0.970443,0.974594,0.966000,0.969201,0.971616,0.970443,0.970080
1,rbf_svm,validation,203,0.960591,0.967195,0.955828,0.959874,0.962413,0.960591,0.960102
2,mlp,validation,203,0.950739,0.960192,0.945657,0.950563,0.953616,0.950739,0.950156
3,logistic_regression,validation,203,0.945813,0.954143,0.939407,0.943518,0.949147,0.945813,0.944760
